# Tidemark: context and memory management Colab notebook

This is the fourth project notebook, alongside `01_toolsmith.ipynb`, `02_steadyhand.ipynb`, and `03_bulkhead.ipynb`. Tidemark will own prefix-stable context assembly, restorable observation shaping, citation-checked compaction, and governed long-term memory. The implementation plan is the eight separate documents in `Tidemark/`.

**Parts 1–2 checkpoint:** the measurement baseline and prefix-stable context are implemented, and later parts are not implemented yet. The CPU gates replay the existing Steadyhand goldens, validate the long-horizon seed set, and run the planted PrefixLint suite. The live cells after it detect a hardware envelope, fit llama-server for Qwen3-8B, check tokenizer parity and the cache counters, run a utilization smoke and the naive append/truncate baseline, and write `steps.csv` and `prefill_by_step.svg` from the records. The prefix smoke then checks the live template against the production prompt, catches planted cache breakers, and measures warm same-slot reuse with the prefix-stable strategy. Nothing above the live cells runs a model, and no cache-hit, latency, or task-success value exists until those cells have run on both a T4 and an L4 and written their manifests.

Run top to bottom on a fresh Colab T4 or L4 runtime. Capacity comes from detected VRAM, vCPUs, RAM, and the fit probe, never from the GPU name alone, and the selected shape stays fixed for the scored baseline. The four setup cells match Toolsmith's canonical cells, the provisioning cell matches Steadyhand's, and cleanup matches Steadyhand's. Records go to `MyDrive/toolsmith/records/<gpu>/tidemark/`; local active journals and samples live under `/content/tidemark/` and are mirrored to Drive.

Put `HF_TOKEN` and `LOCAL_KEY` in `MyDrive/toolsmith/.env` (or `.env.txt`) or Colab Secrets, never in a cell. After a disconnect, reconnect and run all cells from the top: completed baseline units are skipped and an interrupted episode resumes from its journal.

`tests/test_notebook_04.py` checks notebook structure, shared cells, and the CPU gate. The source project brief is [Tidemark: context and memory management for a self-hosted agent](https://agent-track-day-4-of-10-nov-10-2026-4gdn.pages.grok.bot/).

In [ ]:
# Standard-library imports used throughout, and a first look at the GPU and Python version.
# Runs before anything else; it must work even on a CPU runtime.
import json
import os
import platform
import shlex
import shutil
import socket
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

# No check=True: on a CPU runtime the hardware cell explains how to switch instead.
gpu_result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
    capture_output=True,
    text=True,
) if shutil.which('nvidia-smi') else None
gpu_info = gpu_result.stdout.strip() if gpu_result is not None and gpu_result.returncode == 0 else 'No NVIDIA GPU visible'
print(gpu_info)
print(f'Python {platform.python_version()}')

In [ ]:
# Clone or update the repository, mount Google Drive, create the shared cache folders, and load
# HF_TOKEN and LOCAL_KEY from MyDrive/toolsmith/.env or Colab Secrets (never from a cell).
from google.colab import drive, userdata

REPO_URL = 'https://github.com/shreyasnarahari/bare-metal-agent'
PROJECT_DIR = Path('/content/bare-metal-agent')
if (PROJECT_DIR / '.git').is_dir():
    subprocess.run(
        ['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only', 'origin', 'main'],
        check=True,
    )
elif PROJECT_DIR.exists():
    raise FileNotFoundError(
        f'{PROJECT_DIR} exists but is not a bare-metal-agent git checkout.'
    )
else:
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_DIR)], check=True)
if not (PROJECT_DIR / 'pyproject.toml').is_file():
    raise FileNotFoundError(f'Expected {PROJECT_DIR / "pyproject.toml"} after repository setup.')
src_dir = PROJECT_DIR / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

drive.mount('/content/drive')
from toolsmith.provision import read_env_file

DRIVE_ROOT = Path('/content/drive/MyDrive/toolsmith')
DRIVE_BUILD_CACHE = DRIVE_ROOT / 'builds'
DRIVE_MODEL_CACHE = DRIVE_ROOT / 'models'
# Builds (keyed by compute capability) and models are shared by every GPU; records are per GPU.
for directory in (DRIVE_BUILD_CACHE, DRIVE_MODEL_CACHE):
    directory.mkdir(parents=True, exist_ok=True)

env_path = next(
    (
        candidate
        for candidate in (DRIVE_ROOT / '.env', DRIVE_ROOT / '.env.txt')
        if candidate.is_file()
    ),
    DRIVE_ROOT / '.env',
)
env_values = read_env_file(env_path)
def load_secret(name):
    value = env_values.get(name, '').strip()
    if value:
        return value, env_path
    return userdata.get(name), 'Colab Secrets'

HF_TOKEN, hf_source = load_secret('HF_TOKEN')
LOCAL_KEY, local_source = load_secret('LOCAL_KEY')
if not HF_TOKEN or not LOCAL_KEY:
    raise ValueError(f'HF_TOKEN and LOCAL_KEY must be set in {env_path} or Colab Secrets.')
print(f'HF_TOKEN loaded from {hf_source}; LOCAL_KEY loaded from {local_source}')

In [ ]:
# Install this repository (editable, with the Colab extras) into the notebook's Python.
get_ipython().run_line_magic(
    'pip',
    f"install -q -e {shlex.quote(str(PROJECT_DIR) + '[colab]')}",
)

In [ ]:
# Detect the GPU, vCPUs, RAM, and disk once; every later size (build jobs, slots, pools) comes from
# HARDWARE. Also sets the per-GPU records folder, the background-job runner, and the record stamp.
import os
import uuid
from functools import partial
from pathlib import Path

from toolsmith.background import BackgroundJobs
from toolsmith.drive import ensure_drive, sync_drive
from toolsmith.hardware import detect_hardware, require_supported, runtime_stamp

# Every build, server, and pool size below comes from what this runtime has (T4, L4, or other).
HARDWARE = detect_hardware()
print(HARDWARE.summary())
require_supported(HARDWARE)
GPU_FAMILY = HARDWARE.family
CPU_COUNT = HARDWARE.cpu_count
# A T4 run and an L4 run never share a checkpoint, report, or JSONL file.
DRIVE_RECORDS = DRIVE_ROOT / 'records' / HARDWARE.records_slug
DRIVE_RECORDS.mkdir(parents=True, exist_ok=True)
print(f'Records for this GPU: {DRIVE_RECORDS}')
# Network calls give up instead of hanging: git aborts a transfer under 1 KB/s for a minute, and
# Hugging Face requests time out (read when huggingface_hub is first imported, after this cell).
for name, value in {
    'GIT_HTTP_LOW_SPEED_LIMIT': '1000', 'GIT_HTTP_LOW_SPEED_TIME': '60',
    'HF_HUB_DOWNLOAD_TIMEOUT': '60', 'HF_HUB_ETAG_TIMEOUT': '30',
}.items():
    os.environ.setdefault(name, value)
# Downloads, builds, and Drive copies run here while the GPU works; each job logs to its own file.
if 'BACKGROUND' not in globals():
    BACKGROUND = BackgroundJobs(Path(os.environ.get('TOOLSMITH_BACKGROUND_LOGS', '/content/background-logs')))
# Added to every Toolsmith record: the GPU, how busy the machine was, and this kernel's session, so
# records rewritten by a rerun after a disconnect can be told apart from the first run's.
if 'SESSION_ID' not in globals():
    SESSION_ID = uuid.uuid4().hex[:12]
RUNTIME_STAMP = partial(runtime_stamp, HARDWARE, BACKGROUND, session_id=SESSION_ID)

## Eight implementation gates

Stop at each acceptance gate before moving to the next. Each document names its concrete modules, journal contracts, tests, notebook cells, and pitfalls.

| Part | Plan | Notebook outcome |
|---|---|---|
| 1 | `01-baseline-and-context-contract.md` | T4/L4 capacity plans, shared scheduler, utilization smoke, naive baseline, tokenizer/cache/TTFT telemetry |
| 2 | `02-prefix-stable-context-and-budgets.md` | Deterministic five-segment assembler and planted PrefixLint smoke |
| 3 | `03-observation-shaping-and-blob-store.md` | 30–50-output must-keep corpus and blob round trips |
| 4 | `04-cited-compaction-and-replay.md` | Real constrained summaries and before/after source-check evidence |
| 5 | `05-long-term-memory-and-retrieval.md` | SQLite/FTS5/local-embedding cross-session smoke |
| 6 | `06-secure-memory-tools-and-session-resume.md` | Tainted-write approvals, no-effect replay, logical resume; optional KV restore |
| 7 | `07-long-horizon-evaluation-and-ablations.md` | Seven paired modes plus LongMemEval/context-rot adapters |
| 8 | `08-live-runs-results-and-handoff.md` | T4/L4 autotuned sweeps, CPU/GPU overlap, utilization evidence, generated `RESULTS.md` |

The core compatibility rule is that `context=None` leaves existing Steadyhand episodes unchanged. Every model request continues through Toolsmith, every tool continues through Steadyhand and Bulkhead, and replay never repeats persistent memory or external execution effects.

**Phase delivery:** after all code, tests, and required T4/L4 acceptance gates pass, the agent automatically creates an implementation commit, rewrites the canonical `Tidemark/Handoff.md`, creates a Handoff commit containing the implementation SHA, and pushes both to `origin/main` with remote-SHA verification. A fresh session reads that Handoff before the next plan; no previous chat or surviving Colab kernel is required. Failed gates or publication block the next phase. Unrelated dirty files, generated results, credentials, and private artifacts are never staged.

In [ ]:
# Verify the starting point on CPU: the Steadyhand goldens still replay, and the Tidemark seed set validates.
from importlib.util import find_spec

from steadyhand.replay import replay_many
from toolsmith.evaluation.runner import atomic_write_json

tidemark_golden = sorted((Path(PROJECT_DIR) / 'tests' / 'fixtures' / 'steadyhand_episodes').glob('*.jsonl'))
if not tidemark_golden:
    raise RuntimeError('No Steadyhand golden episodes found; cannot establish the Tidemark starting baseline.')
tidemark_workers = int(os.environ.get('TIDEMARK_CPU_GATE_WORKERS', str(min(4, CPU_COUNT))))
tidemark_outcomes = replay_many(tidemark_golden, workers=tidemark_workers)
tidemark_failures = [outcome for outcome in tidemark_outcomes if not outcome.ok]
tidemark_implemented = find_spec('tidemark') is not None
tidemark_dataset = None
if tidemark_implemented:
    from tidemark.eval.baseline import load_long_horizon, validate_long_horizon
    tidemark_dataset = validate_long_horizon(load_long_horizon())
TIDEMARK_STATUS = {
    'event': 'tidemark_cpu_gate',
    'passed': not tidemark_failures and tidemark_implemented,
    'golden_replays': len(tidemark_outcomes),
    'failed_replays': [
        {'path': outcome.path, 'error': outcome.error}
        for outcome in tidemark_failures
    ],
    'implementation_present': tidemark_implemented,
    'implementation_status': 'part_1' if tidemark_implemented else 'missing',
    'seed_dataset': tidemark_dataset,
    'live_results': 'not_run',
    'plan_parts': 8,
    'runtime': RUNTIME_STAMP()['runtime'],
}
tidemark_gate_path = atomic_write_json(DRIVE_RECORDS / 'tidemark' / 'cpu-gate.json', TIDEMARK_STATUS)
if tidemark_failures:
    raise RuntimeError(f'Existing Steadyhand replay baseline failed: {tidemark_failures}')
if not tidemark_implemented:
    raise RuntimeError('The tidemark package is not importable; reinstall the project.')
print(f'Golden replays passed: {len(tidemark_outcomes)}. Seed set: {tidemark_dataset["tasks"]} tasks, '
      f'sha256 {tidemark_dataset["dataset_sha256"][:12]}. Status: {tidemark_gate_path}')
print('No live Tidemark result exists until the cells below run on this GPU.')

In [ ]:
# PrefixLint gate on CPU: the same state renders identically, every planted cache breaker is caught at its exact
# token and segment, and legal appends, tail rewrites, and epoch transitions are not flagged (both Qwen protocols).
from tidemark.assembler import SpanRenderer
from tidemark.eval.prefix import ChatMLTemplate, planted_suite
from tidemark.tokenizer import RegexTokenizer
from toolsmith.evaluation.runner import atomic_write_json
from toolsmith.protocols import QWEN3_PROFILE, QWEN35_PROFILE

TIDEMARK_PREFIX_GATE = {
    profile.name: planted_suite(SpanRenderer(ChatMLTemplate(), RegexTokenizer(), protocol=profile.protocol), repeats=100)
    for profile in (QWEN3_PROFILE, QWEN35_PROFILE)
}
TIDEMARK_PREFIX_GATE['passed'] = all(result['passed'] for result in TIDEMARK_PREFIX_GATE.values())
atomic_write_json(DRIVE_RECORDS / 'tidemark' / 'prefix-gate.json', TIDEMARK_PREFIX_GATE)
if not TIDEMARK_PREFIX_GATE['passed']:
    failed = {name: [case['case'] for case in result['cases'] if not case['passed']]
              for name, result in TIDEMARK_PREFIX_GATE.items() if name != 'passed'}
    raise RuntimeError(f'PrefixLint gate failed: {failed}')
print('PrefixLint gate passed (CPU fake template and tokenizer):', {
    name: [(case['case'], case['status'], case['first_divergent_index']) for case in result['cases']]
    for name, result in TIDEMARK_PREFIX_GATE.items() if name != 'passed'
})

## Part 1 live run: capacity, parity, utilization, baseline

The cells below run in order on one GPU. Run the notebook once on a T4 and once on an L4, with the same model and task ids. Each cell writes raw records first and prints a summary second.

1. **Settings** pick the primary model (Qwen3-8B on both GPUs) and paths, then the shared Steadyhand cell provisions llama.cpp, the GGUF, and the renderer in the background.
2. **Capacity probe** detects the hardware envelope, rejects candidates the estimates say cannot fit, fits llama-server through the KV/flash ladder (shrinking slots, never per-slot context, on OOM), and starts one shared scheduler and resource sampler.
3. **Tokenizer and cache probe** compares the model tokenizer with llama-server's `/tokenize` on the parity corpus and runs a repeated-prompt probe to learn what `prompt_n` and `tokens_evaluated` mean on this build. A parity mismatch stops the notebook; cache metrics stay null unless the probe confirms them.
4. **Utilization smoke** runs the short seed tasks through the scheduler with resource sampling; a miss is explained from the samples, never filled with artificial work.
5. **Baseline** runs every seed task under the naive append/truncate strategy with the shape frozen, forced-prefix steps, and buffered-stream TTFT. It resumes after a disconnect.
6. **Report** writes `steps.csv`, `prefill_by_step.svg`, `summary.json`, and `report.md` from the records of every GPU found on Drive; GPUs and tasks without records are `untested`.
7. **Prefix smoke** (part 2) requires the lint prompt to equal the production prompt, runs the planted cache breakers with the model's template, shows a planted timestamp breaking same-slot reuse while a legal append keeps it, then runs the seed tasks with the prefix-stable strategy and reports warm reuse against the 90% target. Any PrefixLint violation in the live run stops the notebook.

In [ ]:
# Tidemark part 1 settings: the primary model (Qwen3-8B on both GPUs), the TTFT generation path, and the
# local and Drive record folders. Runs before the shared provisioning cell, which reads STEADYHAND_PHASE.
os.environ.setdefault('STEADYHAND_PHASE', 'qwen3/wide')
if not os.environ['STEADYHAND_PHASE'].startswith('qwen3/'):
    raise ValueError('The core Tidemark suite runs Qwen3-8B on both GPUs; set STEADYHAND_PHASE=qwen3/wide or qwen3/tall.')
TIDEMARK_SHAPE = os.environ['STEADYHAND_PHASE'].split('/')[1]
TIDEMARK_LOCAL = Path(os.environ.get('TIDEMARK_LOCAL', '/content/tidemark'))
TIDEMARK_RECORDS = DRIVE_RECORDS / 'tidemark'
# buffered_stream measures TTFT inside Toolsmith; every row of one comparison uses the same path.
TIDEMARK_GENERATION_PATH = os.environ.get('TIDEMARK_GENERATION_PATH', 'buffered_stream')
TIDEMARK_MAX_TOKENS = int(os.environ.get('TIDEMARK_MAX_TOKENS', '512'))
# pinned keeps an episode on one slot (cache measurements); work_conserving is a separate throughput policy.
TIDEMARK_AFFINITY = os.environ.get('TIDEMARK_AFFINITY', 'pinned')
TIDEMARK_RUN_ID = os.environ.get('TIDEMARK_BASELINE_RUN_ID', f'baseline-v1-{HARDWARE.records_slug}-{TIDEMARK_SHAPE}')
for tidemark_directory in (TIDEMARK_LOCAL, TIDEMARK_RECORDS):
    tidemark_directory.mkdir(parents=True, exist_ok=True)
print(json.dumps({
    'phase': os.environ['STEADYHAND_PHASE'], 'generation_path': TIDEMARK_GENERATION_PATH, 'affinity': TIDEMARK_AFFINITY,
    'max_tokens': TIDEMARK_MAX_TOKENS, 'run_id': TIDEMARK_RUN_ID, 'local': str(TIDEMARK_LOCAL), 'records': str(TIDEMARK_RECORDS),
}, indent=2))

In [ ]:
# Start provisioning the model STEADYHAND_PHASE names (llama.cpp build, GGUF, renderer) as
# background jobs and return at once; the server cell collects them after the CPU gate.
from toolsmith.prompting import ChatPromptRenderer
from toolsmith.protocols import QWEN3_PROFILE, QWEN35_PROFILE
from toolsmith.provision import (
    LlamaCppRuntime,
    cache_dir_for,
    ensure_gguf,
    ensure_llama_server,
    q4_k_m_language_file,
    resolve_gguf,
)

STEADYHAND_MODELS = {
    'qwen3': {'profile': QWEN3_PROFILE, 'llama_cpp_ref': 'b5401',
              'repo': 'Qwen/Qwen3-8B-GGUF', 'filename': 'Qwen3-8B-Q4_K_M.gguf'},
    'qwen35': {'profile': QWEN35_PROFILE, 'llama_cpp_ref': 'b8233',
               'repo': 'unsloth/Qwen3.5-9B-GGUF', 'filename': None},
}
steadyhand_profile_name = os.environ.get('STEADYHAND_PHASE', 'qwen35/wide').split('/')[0]
if steadyhand_profile_name not in STEADYHAND_MODELS:
    raise ValueError(f'STEADYHAND_PHASE must start with one of {sorted(STEADYHAND_MODELS)}')
steadyhand_model_spec = STEADYHAND_MODELS[steadyhand_profile_name]
if globals().get('STEADYHAND_ACTIVE'):
    raise RuntimeError('A Steadyhand llama-server is running; run the stop cell before switching models')

GPU_NAME = HARDWARE.gpu_name
GPU_COMPUTE_CAP = HARDWARE.compute_cap
steadyhand_ref = steadyhand_model_spec['llama_cpp_ref']
steadyhand_llama_runtime = LlamaCppRuntime(
    ref=steadyhand_ref,
    source=Path(f'/content/llama.cpp-{steadyhand_ref}-source'),
    build=Path(f'/content/llama.cpp-{steadyhand_ref}-build'),
    cache=cache_dir_for(DRIVE_BUILD_CACHE, steadyhand_ref, GPU_COMPUTE_CAP),
    compute_cap=GPU_COMPUTE_CAP,
    staging=Path(f'/content/llama.cpp-{steadyhand_ref}-staging'),
)

def provision_steadyhand_gguf(spec, profile_name):
    if spec['filename']:
        gguf = resolve_gguf(spec['repo'], token=HF_TOKEN, filename=spec['filename'])
        drive_name = gguf.filename
    else:
        gguf = resolve_gguf(spec['repo'], token=HF_TOKEN, select=q4_k_m_language_file)
        drive_name = f'{gguf.revision[:12]}-{Path(gguf.filename).name}'
    # The same Drive names the Toolsmith notebook uses, so both notebooks share one cached copy.
    local = ensure_gguf(
        gguf,
        local_path=Path('/content') / f'{profile_name}-{Path(gguf.filename).name}',
        drive_path=DRIVE_MODEL_CACHE / drive_name,
        token=HF_TOKEN,
        cache_dir=f'/content/gguf-download-{profile_name}',
        defer=BACKGROUND.submit,
    )
    return gguf, local

# Started here and collected by the server cell, so the CPU gate below runs while the weights
# download or copy from Drive and llama.cpp restores or builds.
BACKGROUND.submit(
    f'steadyhand-llama.cpp-{steadyhand_ref}', ensure_llama_server, steadyhand_llama_runtime,
    jobs=HARDWARE.build_jobs(), defer=BACKGROUND.submit,
)
BACKGROUND.submit(f'steadyhand-gguf-{steadyhand_profile_name}', provision_steadyhand_gguf, steadyhand_model_spec, steadyhand_profile_name)
BACKGROUND.submit(
    f'steadyhand-renderer-{steadyhand_profile_name}', ChatPromptRenderer.from_pretrained,
    steadyhand_model_spec['profile'].tokenizer, token=HF_TOKEN,
)
STEADYHAND_PROVISIONING = {
    'profile_name': steadyhand_profile_name,
    'profile': steadyhand_model_spec['profile'],
    'llama_cpp_ref': steadyhand_ref,
    'build': steadyhand_llama_runtime.build,
    'gpu': GPU_NAME,
    'compute_cap': GPU_COMPUTE_CAP,
}
STEADYHAND_RUNTIME = None

def collect_steadyhand_runtime():
    provisioning = STEADYHAND_PROVISIONING
    name, ref = provisioning['profile_name'], provisioning['llama_cpp_ref']
    server_bin = BACKGROUND.wait(f'steadyhand-llama.cpp-{ref}')
    gguf, local_model = BACKGROUND.wait(f'steadyhand-gguf-{name}')
    return {
        **provisioning,
        'server_bin': server_bin,
        'model': local_model,
        'model_repo': gguf.repo,
        'model_file': gguf.filename,
        'model_revision': gguf.revision,
        'model_size': gguf.size,
        'renderer': BACKGROUND.wait(f'steadyhand-renderer-{name}'),
    }

print(f'Provisioning {steadyhand_profile_name} on llama.cpp {steadyhand_ref} in the background: {BACKGROUND.status()}')

In [ ]:
# Detect the hardware envelope, plan capacity for Qwen3-8B, and fit llama-server with the first candidate whose
# estimate fits (KV/flash ladder; slots shrink on OOM at the same per-slot context). Starts the shared scheduler
# and resource sampler. Defines stop_steadyhand_runtime, which the shared cleanup cell calls.
import dataclasses
import hashlib
from datetime import datetime, timezone

import httpx

from steadyhand import LiveSettings, build_llama_client
from steadyhand.eval.serving import SELF_TEST_PROMPT, decode_speed, kv_ladder, parse_nvidia_smi
from tidemark.eval.serving import QWEN3_8B, CapacityCache, fit_capacity, plan_capacity
from tidemark.runtime import GpuSampler, HardwareEnvelope, ResourceBudget, ResourceSampler
from tidemark.scheduler import Scheduler
from toolsmith.backends import CompletionTruncatedError, LlamaCppClient, LlamaCppServer
from toolsmith.evaluation.runner import atomic_write_json
from toolsmith.provision import free_port, server_environment

def stop_steadyhand_runtime():
    global TIDEMARK_ACTIVE
    active = globals().get('TIDEMARK_ACTIVE') or {}
    if active.get('sampler') is not None:
        active['sampler'].stop()
    if active.get('scheduler') is not None:
        active['scheduler'].close()
    for name in ('client', 'raw_client'):
        if active.get(name) is not None:
            active[name].close()
    if active.get('server') is not None:
        active['server'].stop(timeout=30)
    TIDEMARK_ACTIVE = {}
    os.environ.pop('TOOLSMITH_BASE_URL', None)

def tidemark_query_gpu():
    output = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total,memory.used,utilization.gpu', '--format=csv,noheader,nounits'],
        check=True, capture_output=True, text=True,
    ).stdout
    return parse_nvidia_smi(output)[0]

def tidemark_sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(16 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

if not globals().get('STEADYHAND_PROVISIONING'):
    raise RuntimeError('Run the Tidemark settings cell and the shared Steadyhand provisioning cell first')
if not globals().get('STEADYHAND_RUNTIME'):
    STEADYHAND_RUNTIME = collect_steadyhand_runtime()
tidemark_runtime = STEADYHAND_RUNTIME
stop_steadyhand_runtime()
tidemark_env = server_environment(tidemark_runtime['build'])
os.environ['LD_LIBRARY_PATH'] = tidemark_env['LD_LIBRARY_PATH']
tidemark_help = subprocess.run([str(tidemark_runtime['server_bin']), '--help'], capture_output=True, text=True, env=tidemark_env).stdout
BACKGROUND.submit('tidemark-gguf-sha256', tidemark_sha256, tidemark_runtime['model'])
TIDEMARK_ENVELOPE = HardwareEnvelope.detect(hardware=HARDWARE, help_text=tidemark_help, build=tidemark_runtime['llama_cpp_ref'])
tidemark_budget = ResourceBudget.from_envelope(TIDEMARK_ENVELOPE)
tidemark_model = QWEN3_8B.with_gguf_size(tidemark_runtime['model_size'])
tidemark_log = Path('/content/tidemark-server.log')
tidemark_port = free_port('127.0.0.1', 8090)

def start_tidemark_server(preset, extra_args):
    server = LlamaCppServer(
        executable=tidemark_runtime['server_bin'], model=tidemark_runtime['model'], api_key=LOCAL_KEY,
        log_path=tidemark_log, port=tidemark_port, context_size=preset.context_size,
        gpu_layers=99, parallel=preset.parallel, extra_args=extra_args,
    )
    probe = LlamaCppClient(server.base_url, LOCAL_KEY, timeout=30.0)
    try:
        server.start()
        server.wait_until_healthy(probe, timeout=900, poll_interval=2)
    except BaseException:
        server.stop(timeout=30)
        raise
    finally:
        probe.close()
    return server

def probe_tidemark_decode(server):
    with LlamaCppClient(server.base_url, LOCAL_KEY, timeout=300.0) as client:
        started = time.perf_counter()
        try:
            response = client.complete(SELF_TEST_PROMPT, max_tokens=64, temperature=0.0, seed=0)
        except CompletionTruncatedError as error:
            response = error.response
        return decode_speed(response, time.perf_counter() - started)

# Every rung's estimate is recorded, including the ones rejected before launch.
tidemark_decisions = [
    plan_capacity(TIDEMARK_ENVELOPE, tidemark_model, shape=TIDEMARK_SHAPE, kv_cache=rung['kv_cache'],
                  flash_attn=rung['flash_attn'], affinity=TIDEMARK_AFFINITY, budget=tidemark_budget)
    for rung in kv_ladder(tidemark_help)
]
tidemark_candidate = next((decision.plan for decision in tidemark_decisions if decision.plan is not None), None)
tidemark_capacity_record = {
    'event': 'tidemark_capacity', 'recorded_at': datetime.now(timezone.utc).isoformat(),
    'envelope': TIDEMARK_ENVELOPE.as_dict(), 'budget': tidemark_budget.as_dict(),
    'decisions': [decision.as_dict() for decision in tidemark_decisions], 'runtime': RUNTIME_STAMP()['runtime'],
}
if tidemark_candidate is None:
    atomic_write_json(TIDEMARK_RECORDS / 'capacity' / f'{TIDEMARK_SHAPE}-did-not-fit.json', tidemark_capacity_record)
    raise RuntimeError(f'did_not_fit: no {TIDEMARK_SHAPE} candidate fits this GPU even at one slot; see the capacity record')
tidemark_fit, TIDEMARK_PLAN = fit_capacity(
    tidemark_candidate, help_text=tidemark_help, start=start_tidemark_server,
    query_gpu=tidemark_query_gpu, probe=probe_tidemark_decode,
)
os.environ['TOOLSMITH_BASE_URL'] = tidemark_fit.server.base_url
CapacityCache(TIDEMARK_RECORDS / 'capacity' / 'plans.json').put(
    CapacityCache.key(TIDEMARK_ENVELOPE.hardware_key(), tidemark_runtime['llama_cpp_ref'], tidemark_model, {'shape': TIDEMARK_SHAPE}),
    TIDEMARK_PLAN,
)
# Measured llama.cpp capabilities: /slots answers, and a request pinned to slot 0 reports slot 0.
with httpx.Client(base_url=tidemark_fit.server.base_url, headers={'Authorization': f'Bearer {LOCAL_KEY}'}, timeout=30) as http:
    tidemark_slots_endpoint = http.get('/slots').status_code == 200
tidemark_raw_client = LlamaCppClient(tidemark_fit.server.base_url, LOCAL_KEY, timeout=300.0)
try:
    tidemark_pinned = tidemark_raw_client.complete('Say OK.', max_tokens=2, temperature=0.0, seed=0, id_slot=0)
except CompletionTruncatedError as error:
    tidemark_pinned = error.response
TIDEMARK_ENVELOPE = dataclasses.replace(TIDEMARK_ENVELOPE, llama=TIDEMARK_ENVELOPE.llama.with_probe(
    slots_endpoint=tidemark_slots_endpoint, id_slot=tidemark_pinned.get('id_slot') == 0,
))
tidemark_scheduler = Scheduler(TIDEMARK_PLAN.scheduler_plan())
tidemark_run_dir = TIDEMARK_LOCAL / 'baseline' / TIDEMARK_RUN_ID
# The sampler reads whichever scheduler is current, so the autotune candidates are sampled too.
tidemark_sampler = ResourceSampler(
    tidemark_run_dir / 'resource_samples.jsonl', interval=0.5, snapshot=lambda: TIDEMARK_ACTIVE['scheduler'].snapshot(),
    gpu=GpuSampler(),
    pid_groups=lambda: {'llama_server': [tidemark_fit.server.process.pid] if tidemark_fit.server.process is not None else []},
)
TIDEMARK_ACTIVE = {
    'server': tidemark_fit.server, 'plan': TIDEMARK_PLAN, 'fit_attempts': tidemark_fit.attempts, 'gpu': tidemark_fit.gpu,
    'decode_tps': tidemark_fit.decode_tps, 'scheduler': tidemark_scheduler, 'sampler': tidemark_sampler,
    'raw_client': tidemark_raw_client, 'profile': tidemark_runtime['profile'], 'renderer': tidemark_runtime['renderer'],
    'client': build_llama_client(LiveSettings.from_env(api_key=LOCAL_KEY), connections=TIDEMARK_PLAN.total_slots + 2, timeout=900.0),
}
tidemark_sampler.start()
tidemark_capacity_record.update(
    plan=TIDEMARK_PLAN.as_dict(), plan_sha256=TIDEMARK_PLAN.sha256(), fit_attempts=tidemark_fit.attempts,
    llama=TIDEMARK_ENVELOPE.as_dict()['llama'], decode_tps=tidemark_fit.decode_tps,
)
atomic_write_json(TIDEMARK_RECORDS / 'capacity' / f'{TIDEMARK_SHAPE}-{TIDEMARK_PLAN.sha256()[:12]}.json', tidemark_capacity_record)
print(json.dumps({key: tidemark_capacity_record[key] for key in ('plan', 'fit_attempts', 'llama', 'decode_tps')}, indent=2, default=str))

In [ ]:
# Tokenizer parity against the running llama-server, then the repeated-prompt cache probe. A parity mismatch
# blocks every token-based number (there is no character-count fallback); unconfirmed cache counters stay null.
from tidemark.telemetry import probe_cache_semantics, slot_completion
from tidemark.tokenizer import CachedTokenizer, HFTokenizer, ServerTokenizer, parity_corpus, run_parity

if not globals().get('TIDEMARK_ACTIVE'):
    raise RuntimeError('Run the Tidemark capacity probe first')
tidemark_renderer = TIDEMARK_ACTIVE['renderer']
tidemark_local_tokenizer = HFTokenizer(tidemark_renderer.tokenizer, name=TIDEMARK_ACTIVE['profile'].tokenizer)
tidemark_server_tokenizer = ServerTokenizer(TIDEMARK_ACTIVE['server'].base_url, LOCAL_KEY)
TIDEMARK_SERVING = {
    'model_repo': STEADYHAND_RUNTIME['model_repo'], 'model_file': STEADYHAND_RUNTIME['model_file'],
    'model_revision': STEADYHAND_RUNTIME['model_revision'], 'gguf_sha256': BACKGROUND.wait('tidemark-gguf-sha256'),
    'llama_cpp': STEADYHAND_RUNTIME['llama_cpp_ref'], 'per_slot_context': TIDEMARK_PLAN.per_slot_context,
    'parallel': TIDEMARK_PLAN.total_slots, 'kv_cache': TIDEMARK_PLAN.kv_cache, 'flash_attn': TIDEMARK_PLAN.flash_attn,
    'cache_prompt': 'server_default', 'slot_save': None,
}
try:
    tidemark_parity = run_parity(
        tidemark_local_tokenizer, tidemark_server_tokenizer,
        parity_corpus(lambda messages, tools: tidemark_renderer.render(messages, tools, enable_thinking=False)),
        identity={**tidemark_local_tokenizer.identity(), **TIDEMARK_SERVING},
    )
finally:
    tidemark_server_tokenizer.close()
atomic_write_json(TIDEMARK_RECORDS / 'parity' / f'{TIDEMARK_RUN_ID}.json', tidemark_parity.as_dict())
if not tidemark_parity.passed:
    raise RuntimeError(f'Tokenizer parity failed on {tidemark_parity.mismatches}; token-based acceptance is blocked.')
TIDEMARK_TOKENIZER = CachedTokenizer(tidemark_local_tokenizer, tidemark_parity)
tidemark_probe_prompt = tidemark_renderer.render(
    [{'role': 'system', 'content': 'You are a support agent.'}, {'role': 'user', 'content': 'Is order ord_1001 delivered?'}],
    enable_thinking=False,
)
TIDEMARK_CACHE = probe_cache_semantics(
    slot_completion(TIDEMARK_ACTIVE['raw_client'], id_slot=0), tidemark_probe_prompt,
    lambda text: len(tidemark_local_tokenizer.encode(text)),
)
atomic_write_json(TIDEMARK_RECORDS / 'parity' / f'{TIDEMARK_RUN_ID}-cache-probe.json', TIDEMARK_CACHE.as_dict())
print(json.dumps({'parity_passed': tidemark_parity.passed, 'cases': len(tidemark_parity.results),
                  'cache_semantics': TIDEMARK_CACHE.as_dict()}, indent=2))

In [ ]:
# Unscored autotune and utilization smoke: the short seed tasks run through the shared scheduler once per
# episode-worker candidate while the sampler records slot, queue, CPU, RAM, and GPU use. The candidate with the most
# correct completed tasks per minute is frozen for the scored baseline. A miss against the 95% slot / 85% CPU targets
# is explained from the samples, never filled with artificial work. Affinity stays TIDEMARK_AFFINITY throughout.
import dataclasses
from datetime import datetime, timezone

from steadyhand import SteadyhandClient, worker_service
from tidemark.baseline import NaiveAppendTruncate
from tidemark.eval.baseline import (
    BaselineSpec, ForcedPrefixClient, dataset_sha256, load_baseline_scores, load_long_horizon, run_baseline,
)
from tidemark.eval.serving import TuneMeasurement, autotune, utilization_smoke
from tidemark.scheduler import Scheduler
from tidemark.types import ContextConfig
from toolsmith.evaluation.runner import atomic_write_json

if not globals().get('TIDEMARK_TOKENIZER'):
    raise RuntimeError('Run the tokenizer and cache probe first')
TIDEMARK_TASKS = load_long_horizon()
TIDEMARK_CONTEXT = ContextConfig(
    slot_context=TIDEMARK_PLAN.per_slot_context, max_tokens=TIDEMARK_MAX_TOKENS, tokenizer=TIDEMARK_TOKENIZER.identity(),
    template_sha256=TIDEMARK_TOKENIZER.identity().get('template_sha256'), generation_path=TIDEMARK_GENERATION_PATH,
    cache_semantics=TIDEMARK_CACHE.config(),
)

def tidemark_client(task, owner):
    service = worker_service(TIDEMARK_ACTIVE['client'], TIDEMARK_ACTIVE['renderer'], TIDEMARK_ACTIVE['profile'],
                             generation_path=TIDEMARK_GENERATION_PATH)
    live = SteadyhandClient(service, max_tokens=TIDEMARK_MAX_TOKENS, seed=task.seed, slot_context=TIDEMARK_PLAN.per_slot_context)
    return ForcedPrefixClient(live, task)

def tidemark_session(task, client):
    return NaiveAppendTruncate(
        TIDEMARK_CONTEXT, render=lambda messages, tools, choice: client.render_prompt(messages, tools, tool_choice=choice),
        tokenizer=TIDEMARK_TOKENIZER,
    )

def tidemark_spec(run_id, tasks, plan):
    return BaselineSpec(
        run_id=run_id, task_ids=tuple(task.task_id for task in tasks), dataset_sha256=dataset_sha256(tasks),
        context_config=TIDEMARK_CONTEXT.as_dict(), mode='forced_prefix', gpu=HARDWARE.records_slug,
        model={'profile': TIDEMARK_ACTIVE['profile'].name, 'max_tokens': TIDEMARK_MAX_TOKENS, 'temperature': 0.0},
        server={**TIDEMARK_SERVING, 'plan_sha256': plan.sha256()}, scheduler=plan.scheduler_plan().as_dict(),
    )

tidemark_smoke_tasks = [task for task in TIDEMARK_TASKS if task.distance_N <= int(os.environ.get('TIDEMARK_SMOKE_MAX_N', '10'))]
tidemark_stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
tidemark_smoke_results = {}

def tidemark_evaluate(plan):
    TIDEMARK_ACTIVE['scheduler'].close()
    TIDEMARK_ACTIVE['scheduler'] = Scheduler(plan.scheduler_plan())
    run_id = f'smoke-{HARDWARE.records_slug}-{tidemark_stamp}-w{plan.episode_workers}'
    result = utilization_smoke(
        lambda: run_baseline(
            tidemark_spec(run_id, tidemark_smoke_tasks, plan), TIDEMARK_TASKS, client_factory=tidemark_client,
            session_factory=tidemark_session, run_dir=TIDEMARK_LOCAL / 'smoke' / run_id,
            scheduler=TIDEMARK_ACTIVE['scheduler'], secrets=(HF_TOKEN, LOCAL_KEY), context_server=TIDEMARK_SERVING,
        ),
        scheduler=TIDEMARK_ACTIVE['scheduler'], sampler=TIDEMARK_ACTIVE['sampler'],
    )
    scores = load_baseline_scores(TIDEMARK_LOCAL / 'smoke' / run_id / f'{run_id}.scores.jsonl')
    outcomes = [attempts[-1].outcome for attempts in scores.values()]
    correct = bool(outcomes) and all(outcome in ('success', 'failure') for outcome in outcomes)
    stats = result['slot_stats']
    tidemark_smoke_results[plan.sha256()] = {**result, 'run_id': run_id, 'outcomes': outcomes}
    return TuneMeasurement(
        plan.sha256(), fits=True, correct=correct,
        tasks_per_minute=len(outcomes) / (result['elapsed_s'] / 60) if result['elapsed_s'] else None,
        queue_delay_s=stats['total_wait_s'] / stats['leases'] if stats['leases'] else None,
        notes=tuple(result['utilization']['explanations']),
    )

tidemark_worker_options = sorted({TIDEMARK_PLAN.agent_slots + 1, TIDEMARK_PLAN.episode_workers, 2 * TIDEMARK_PLAN.agent_slots})
# Resuming a baseline: confirm its frozen scheduler with one smoke instead of retuning (a new shape needs a new run id).
tidemark_recorded_spec = next((
    path for path in (TIDEMARK_LOCAL / 'baseline' / TIDEMARK_RUN_ID / f'{TIDEMARK_RUN_ID}.json',
                      TIDEMARK_RECORDS / 'baseline' / TIDEMARK_RUN_ID / f'{TIDEMARK_RUN_ID}.json')
    if path.exists()
), None)
if tidemark_recorded_spec is not None:
    tidemark_worker_options = [json.loads(tidemark_recorded_spec.read_text())['scheduler']['episode_workers']]
elif os.environ.get('TIDEMARK_AUTOTUNE', '1') != '1':
    tidemark_worker_options = [TIDEMARK_PLAN.episode_workers]
tidemark_candidates = [dataclasses.replace(TIDEMARK_PLAN, episode_workers=workers) for workers in tidemark_worker_options]
TIDEMARK_FROZEN = autotune(tidemark_candidates, tidemark_evaluate, hardware_key=TIDEMARK_ENVELOPE.hardware_key(),
                           build=STEADYHAND_RUNTIME['llama_cpp_ref'])
TIDEMARK_PLAN = TIDEMARK_FROZEN.plan
TIDEMARK_ACTIVE['scheduler'].close()
TIDEMARK_ACTIVE['scheduler'] = Scheduler(TIDEMARK_PLAN.scheduler_plan())
TIDEMARK_SMOKE = {'frozen': TIDEMARK_FROZEN.as_dict(), 'frozen_sha256': TIDEMARK_FROZEN.sha256(),
                  'selected_smoke': tidemark_smoke_results[TIDEMARK_PLAN.sha256()], 'candidates': tidemark_smoke_results}
atomic_write_json(TIDEMARK_RECORDS / 'smoke' / f'autotune-{HARDWARE.records_slug}-{tidemark_stamp}.json', TIDEMARK_SMOKE)
print(json.dumps({'selected_episode_workers': TIDEMARK_PLAN.episode_workers, 'frozen_sha256': TIDEMARK_FROZEN.sha256(),
                  'utilization': TIDEMARK_SMOKE['selected_smoke']['utilization'],
                  'measurements': [m.as_dict() for m in TIDEMARK_FROZEN.measurements]}, indent=2, default=str))

In [ ]:
# The naive append/truncate baseline on every seed task, shape frozen. Forced-prefix steps keep histories identical
# across modes; TTFT comes from the buffered-stream path. Resumes after a disconnect (the run folder is restored from
# Drive first); completed units are skipped, infrastructure aborts rerun under a new label.
from tidemark.runtime import read_samples, summarize_utilization
from toolsmith.artifacts import copy_snapshot
from toolsmith.evaluation.runner import atomic_write_json

if not globals().get('TIDEMARK_SMOKE'):
    raise RuntimeError('Run the utilization smoke first')
tidemark_drive_run = TIDEMARK_RECORDS / 'baseline' / TIDEMARK_RUN_ID

def tidemark_mirror(source, destination):
    for path in sorted(Path(source).rglob('*')):
        if path.is_file() and not path.name.endswith('.claim'):
            copy_snapshot(path, Path(destination) / path.relative_to(source))

if not (tidemark_run_dir / f'{TIDEMARK_RUN_ID}.json').exists() and (tidemark_drive_run / f'{TIDEMARK_RUN_ID}.json').exists():
    tidemark_mirror(tidemark_drive_run, tidemark_run_dir)
TIDEMARK_ACTIVE['sampler'].set_phase('scored')
TIDEMARK_SCORES = run_baseline(
    tidemark_spec(TIDEMARK_RUN_ID, TIDEMARK_TASKS, TIDEMARK_PLAN), TIDEMARK_TASKS, client_factory=tidemark_client,
    session_factory=tidemark_session, run_dir=tidemark_run_dir, scheduler=TIDEMARK_ACTIVE['scheduler'],
    secrets=(HF_TOKEN, LOCAL_KEY), context_server=TIDEMARK_SERVING,
    on_scored=lambda score: BACKGROUND.submit(f'tidemark-mirror-{score.task_id}-{score.attempt}', tidemark_mirror, tidemark_run_dir, tidemark_drive_run),
)
TIDEMARK_ACTIVE['sampler'].set_phase('drain')
# Utilization over the scored phase only, with the scheduler that produced it (the throughput rationale).
TIDEMARK_UTILIZATION = {
    'run_id': TIDEMARK_RUN_ID, 'frozen_sha256': TIDEMARK_FROZEN.sha256(), 'scheduler': TIDEMARK_PLAN.scheduler_plan().as_dict(),
    'slot_stats': TIDEMARK_ACTIVE['scheduler'].slots.stats(),
    'utilization': summarize_utilization(read_samples(TIDEMARK_ACTIVE['sampler'].path), phases=('scored',)),
}
atomic_write_json(tidemark_run_dir / 'utilization.json', TIDEMARK_UTILIZATION)
tidemark_mirror(tidemark_run_dir, tidemark_drive_run)
print(json.dumps(TIDEMARK_UTILIZATION, indent=2, default=str))
print(json.dumps([{key: score.as_dict()[key] for key in ('task_id', 'outcome', 'evidence_available', 'steps', 'error_code')}
                  for score in TIDEMARK_SCORES], indent=2))

In [ ]:
# Generate steps.csv, prefill_by_step.svg, summary.json, and report.md from the raw records of every GPU found on
# Drive. GPUs and tasks without records are marked untested; nothing is typed in by hand.
from tidemark.eval.report import write_report

tidemark_runs = {
    gpu: DRIVE_ROOT / 'records' / gpu / 'tidemark' / 'baseline' / f'baseline-v1-{gpu}-{TIDEMARK_SHAPE}'
    for gpu in ('t4', 'l4')
}
tidemark_runs[HARDWARE.records_slug] = tidemark_run_dir
tidemark_report_dir = TIDEMARK_LOCAL / 'report' / TIDEMARK_SHAPE
TIDEMARK_REPORT = write_report(tidemark_runs, tidemark_report_dir)
tidemark_mirror(tidemark_report_dir, TIDEMARK_RECORDS / 'report' / TIDEMARK_SHAPE)
print((tidemark_report_dir / 'report.md').read_text())
print({name: str(path) for name, path in TIDEMARK_REPORT.items()})

In [ ]:
# Prefix-stable context, live: the live template's lint prompt must equal the production prompt; planted breakers
# must be caught with the model's template and tokenizer; on one slot, a planted timestamp must force a near-full
# prefill while a legal append reuses the cache. Only then are the seed tasks run with the prefix-stable strategy,
# pinned slots, and the frozen shape. The warm, non-cold steps' reuse is reported against the 90% target (a target,
# not a claim). Renderer, tokenizer, and grammar cache counters are overhead, never KV-cache hits.
from dataclasses import replace
from statistics import median

from steadyhand import SteadyhandClient, build_registry, worker_service
from steadyhand.journal import read_journal
from tidemark.assembler import ContextAssembler, PrefixStableSession, SpanRenderer
from tidemark.cache import ContentCache
from tidemark.eval.baseline import ForcedPrefixClient, run_baseline
from tidemark.eval.prefix import base_state, planted_prompts, planted_suite
from tidemark.runtime import read_samples, summarize_utilization
from tidemark.telemetry import slot_completion
from tidemark.types import ContextConfig
from toolsmith.api import ChatCompletionRequest
from toolsmith.evaluation.runner import atomic_write_json

if not globals().get('TIDEMARK_SCORES'):
    raise RuntimeError('Run the part 1 baseline cells first')
tidemark_template_sha256 = TIDEMARK_TOKENIZER.identity().get('template_sha256')
TIDEMARK_PREFIX_CACHE = ContentCache('rendered_prefixes', max_entries=4096)
TIDEMARK_GRAMMARS = ContentCache('grammars', max_entries=256)

def tidemark_span_renderer():
    return SpanRenderer(TIDEMARK_ACTIVE['renderer'], TIDEMARK_TOKENIZER, protocol=TIDEMARK_ACTIVE['profile'].protocol,
                        enable_thinking=False, template_sha256=tidemark_template_sha256, cache=TIDEMARK_PREFIX_CACHE)

# 1. The lint prompt is the production prompt.
tidemark_assembled = ContextAssembler(tidemark_span_renderer()).render(base_state())
tidemark_service = worker_service(TIDEMARK_ACTIVE['client'], TIDEMARK_ACTIVE['renderer'], TIDEMARK_ACTIVE['profile'])
tidemark_production = tidemark_service.render_prompt(ChatCompletionRequest(
    model=TIDEMARK_ACTIVE['profile'].model, messages=list(tidemark_assembled.messages), tools=list(tidemark_assembled.tools),
    chat_template_kwargs={'enable_thinking': False},
))
if tidemark_production != tidemark_assembled.spans.text:
    raise RuntimeError('The linted prompt differs from the production prompt; PrefixLint evidence would be wrong')
if tidemark_assembled.spans.status != 'ok':
    raise RuntimeError(f'span_mapping_unavailable with the live template: {tidemark_assembled.spans.detail}')
# 2. Planted breakers with the model's template and tokenizer.
tidemark_live_suite = planted_suite(tidemark_span_renderer(), repeats=25)
if not tidemark_live_suite['passed']:
    raise RuntimeError(f"Live PrefixLint suite failed: {[c for c in tidemark_live_suite['cases'] if not c['passed']]}")
# 3. Cache effect on one slot: warm with the base prompt, then a planted breaker, re-warm, then a legal append.
tidemark_prompts = planted_prompts(tidemark_span_renderer())
tidemark_slot = slot_completion(TIDEMARK_ACTIVE['raw_client'], id_slot=0)
tidemark_cache_probe = []
for name in ('base', 'base', 'timestamp_in_frozen', 'base', 'recent_append'):
    rendered = len(TIDEMARK_TOKENIZER.encode(tidemark_prompts[name]))
    timings = tidemark_slot(tidemark_prompts[name]).get('timings') or {}
    tidemark_cache_probe.append({'prompt': name, 'rendered': rendered, 'prompt_n': timings.get('prompt_n'),
                                 'prompt_ms': timings.get('prompt_ms')})

# 4. The seed tasks with the prefix-stable strategy, shape frozen, pinned slots.
TIDEMARK_PREFIX_CONTEXT = ContextConfig(
    slot_context=TIDEMARK_PLAN.per_slot_context, max_tokens=TIDEMARK_MAX_TOKENS, strategy='prefix_stable',
    tokenizer=TIDEMARK_TOKENIZER.identity(), template_sha256=tidemark_template_sha256,
    generation_path=TIDEMARK_GENERATION_PATH, cache_semantics=TIDEMARK_CACHE.config(),
)
tidemark_sessions = []

def tidemark_prefix_session(task, client):
    tidemark_sessions.append(PrefixStableSession(TIDEMARK_PREFIX_CONTEXT, tidemark_span_renderer()))
    return tidemark_sessions[-1]

def tidemark_prefix_client(task, owner):
    service = worker_service(TIDEMARK_ACTIVE['client'], TIDEMARK_ACTIVE['renderer'], TIDEMARK_ACTIVE['profile'],
                             generation_path=TIDEMARK_GENERATION_PATH, grammar_cache=TIDEMARK_GRAMMARS)
    live = SteadyhandClient(service, max_tokens=TIDEMARK_MAX_TOKENS, seed=task.seed, slot_context=TIDEMARK_PLAN.per_slot_context)
    return ForcedPrefixClient(live, task)

PrefixStableSession(TIDEMARK_PREFIX_CONTEXT, tidemark_span_renderer()).validate(
    TIDEMARK_TASKS[0].messages(), build_registry().schemas(),
)
tidemark_prefix_id = f'prefix-v1-{HARDWARE.records_slug}-{TIDEMARK_SHAPE}'
tidemark_prefix_dir = TIDEMARK_LOCAL / 'prefix' / tidemark_prefix_id
tidemark_prefix_spec = replace(tidemark_spec(tidemark_prefix_id, TIDEMARK_TASKS, TIDEMARK_PLAN),
                               context_config=TIDEMARK_PREFIX_CONTEXT.as_dict())
TIDEMARK_ACTIVE['sampler'].set_phase('prefix')
TIDEMARK_PREFIX_SCORES = run_baseline(
    tidemark_prefix_spec, TIDEMARK_TASKS, client_factory=tidemark_prefix_client, session_factory=tidemark_prefix_session,
    run_dir=tidemark_prefix_dir, scheduler=TIDEMARK_ACTIVE['scheduler'], secrets=(HF_TOKEN, LOCAL_KEY),
    context_server=TIDEMARK_SERVING,
)
TIDEMARK_ACTIVE['sampler'].set_phase('drain')
# Warm reuse: ordinary (non-cold, non-epoch) steps whose prompt passed PrefixLint, on the episode's pinned slot.
tidemark_warm, tidemark_lint = [], {}
for score in TIDEMARK_PREFIX_SCORES:
    records = read_journal(score.journal_path)
    for index, record in enumerate(records):
        if record['event'] == 'context_render' and record['status'] == 'ok':
            tidemark_lint[record['lint_status']] = tidemark_lint.get(record['lint_status'], 0) + 1
            timings = next((r for r in records[index + 1:index + 3] if r['event'] == 'backend_timings'), None)
            if record['lint_status'] in ('expected_append', 'expected_tail_only') and timings and timings['cache_reuse_fraction'] is not None:
                tidemark_warm.append(timings['cache_reuse_fraction'])
TIDEMARK_PREFIX = {
    'run_id': tidemark_prefix_id, 'live_suite_passed': tidemark_live_suite['passed'],
    'production_equals_lint_prompt': True, 'span_status': tidemark_assembled.spans.status,
    'cache_probe': tidemark_cache_probe, 'lint_statuses': tidemark_lint,
    'warm_reuse_median': median(tidemark_warm) if tidemark_warm else None, 'warm_steps': len(tidemark_warm),
    'warm_reuse_target': 0.90, 'cache_semantics': TIDEMARK_CACHE.config(),
    'outcomes': [score.outcome for score in TIDEMARK_PREFIX_SCORES],
    'overhead': {'segment_cache': TIDEMARK_PREFIX_CACHE.stats(), 'grammar_cache': TIDEMARK_GRAMMARS.stats(),
                 'render_us': sum(s.performance()['render_us'] for s in tidemark_sessions),
                 'tokenize_us': sum(s.performance()['tokenize_us'] for s in tidemark_sessions)},
    'utilization': summarize_utilization(read_samples(TIDEMARK_ACTIVE['sampler'].path), phases=('prefix',)),
}
atomic_write_json(tidemark_prefix_dir / 'prefix-summary.json', TIDEMARK_PREFIX)
tidemark_mirror(tidemark_prefix_dir, TIDEMARK_RECORDS / 'prefix' / tidemark_prefix_id)
if tidemark_lint.get('violation'):
    raise RuntimeError(f"PrefixLint found {tidemark_lint['violation']} violation(s) in the live run; see the journals")
print(json.dumps(TIDEMARK_PREFIX, indent=2, default=str))

## Later parts: full live execution after parts 2–7 land

Part 1's cells above provide the capacity probe, tokenizer and cache probes, a measured episode-worker autotune, the utilization smoke, and the naive baseline. The final notebook adds a seven-mode smoke, paired long-horizon ablations, LongMemEval and context-rot subsets, and logical session resume. The core Qwen3-8B suite runs on both T4 and L4 with the same eligible task ids; Qwen3.5 is a separate optional probe. Optional llama.cpp slot restoration runs only after the pinned server proves support.

**Starting shapes, never hardcoded results:** T4 starts with two agent slots at 16K per slot and a small RAM-bounded CPU/IO pipeline; L4 starts with four at 16K. Tall phases try 32K with fewer slots before reporting `did_not_fit`. Actual VRAM, vCPUs, RAM, KV/flash support, and throughput decide the final shape. Reduce slots before per-slot context, and never truncate a required task to make it fit.

**Autotune, then freeze:** a bounded unscored grid measures KV/flash settings, batch/threads, shared versus dedicated compactor slots, slot affinity, CPU/IO workers, and prefetch by correct completed tasks per minute. Scored runs keep the selected complete configuration fixed. Model calls are capped by slot leases while additional episode workers perform tools and CPU preparation. `pinned` scheduling supports cache experiments; `work_conserving` scheduling is a separate, explicitly labeled throughput policy.

**Useful utilization:** setup downloads/builds overlap CPU gates; agent generation overlaps shaping, source checks, embeddings/indexing, next-task preparation, and asynchronous Drive copies. Shared bounded pools prevent T4 CPU/RAM thrashing and L4 over-allocation. Resource samples target at least 95% median slot occupancy during model-ready windows and 85% allocated-CPU utilization during CPU-ready windows, with any miss explained rather than filled with artificial work. Timing comparisons remain within one frozen GPU configuration.

Cold, ordinary, compaction, and resumed steps stay separate. TTFT comes only from first streamed content; non-streaming elapsed time is not relabeled. Live claims are generated from journals, manifests, and resource samples, never from notebook placeholders or paper targets. Cross-GPU completion requires real evidence from both a T4 and an L4.

See `Tidemark/01-baseline-and-context-contract.md` for the shared scheduler and `Tidemark/08-live-runs-results-and-handoff.md` for the exact cell sequence, T4/L4 candidates, durability sets, utilization metrics, and acceptance criteria.

In [ ]:
# Stop the server, let Drive copies finish, stop background jobs, and flush Drive. Keep last.
if 'stop_steadyhand_runtime' in globals():
    stop_steadyhand_runtime()
if 'BACKGROUND' in globals():
    # Let Drive write-backs finish (they are atomic, so an unfinished one is redone next runtime).
    unfinished = BACKGROUND.drain(timeout=float(os.environ.get('TOOLSMITH_CLEANUP_DRAIN_SECONDS', '600')))
    stopped = BACKGROUND.cancel_all()
    if unfinished:
        print(f'Background jobs not finished cleanly: {unfinished}; logs in {BACKGROUND.log_dir}')
    if stopped:
        print(f'Stopped background jobs: {stopped}')
    # Flush every pending Drive write before the runtime goes away (Drive uploads in the background).
    sync_drive(BACKGROUND, stage='cleanup', remount=False)
print('Steadyhand llama-server stopped; Drive flushed and unmounted. Record Colab units used, then delete or disconnect the runtime.')